# 04. Приоритет складов и индекс «относительного риска 0–100»

Блокнот отвечает на два вопроса по стенду: как устанавливается приоритет и по какой формуле считается индекс на карте.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib
matplotlib.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.3})
import matplotlib.pyplot as plt
import mltj
sites, pts, cov = mltj.load()
print(len(sites), 'складов,', len(pts), 'точек')

## Приоритет склада

Для каждой точки: среднее по Pb, As, Hg, Zn от `log(значение) − log(медиана выборки по веществу)`. Индекс склада — медиана по его точкам; склады ранжируются по убыванию. Rb и Co не включены до заключения геохимика.

In [ ]:
sub=['Pb','As','Hg','Zn']
ex=pd.concat([np.log(pts[f'{e}_mgkg'])-np.log(pts[f'{e}_mgkg']).median() for e in sub],axis=1)
pts['composite']=ex.mean(axis=1)
pr=pts.groupby('site_name').composite.median().sort_values(ascending=False).to_frame('индекс'); pr['ранг']=pr.индекс.rank(ascending=False,method='first').astype(int)
chk=sites.set_index('name').priority_rank
pr['ранг в БД']=pr.index.map(chk)
pr.round(3)

## Индекс 0–100 на карте

1. Модель Pb обучается на всех 169 точках; целевая переменная — отклонение от медианы своего склада.
2. Предсказание для тех же точек переводится в перцентиль среди всех точек и умножается на 100.
3. Уровни: low (0–25), medium (25–50), high (50–75), critical (75–100). Цвет склада — медиана индекса по его точкам.

In [ ]:
y=mltj.target(pts,'Pb'); m=mltj.rf(0).fit(pts[mltj.FEATURES].values,y)
pts['score']=pd.Series(m.predict(pts[mltj.FEATURES].values),index=pts.index).rank(pct=True)*100
pts['level']=pd.cut(pts.score,[-1,25,50,75,101],labels=['low','medium','high','critical'])
print('доли уровней по точкам:'); print(pts.level.value_counts(normalize=True).round(2).to_string())

Доли уровней близки к 25% по построению: индекс является рангом, а не концентрацией и не превышением порога.

## Почему цвет склада по индексу модели не совпадает с приоритетом

Целевая переменная вычитает медиану склада, поэтому различия между складами в неё не входят. Медиана индекса по складу определяется признаками уровня склада (pH) и переобучением на обучающих данных. Поэтому на карте цвет склада задаётся рангом приоритета, а индекс модели показывается как положение внутри склада.

In [ ]:
cmp=pts.groupby('site_name').agg(индекс_модели=('score','median'),индекс_приоритета=('composite','median'))
cmp['ранг приоритета']=cmp.index.map(pr['ранг'])
cmp=cmp.sort_values('ранг приоритета'); cmp.round(2)

In [ ]:
from scipy.stats import spearmanr
print('корреляция Спирмена (индекс модели vs индекс приоритета, по складам): %.2f' % spearmanr(cmp.индекс_модели,cmp.индекс_приоритета)[0])
fig,ax=plt.subplots(figsize=(5,4)); ax.scatter(cmp.индекс_приоритета,cmp.индекс_модели)
for n,r in cmp.iterrows(): ax.annotate(n,(r.индекс_приоритета,r.индекс_модели),fontsize=8,xytext=(3,3),textcoords='offset points')
ax.set_xlabel('индекс приоритета (между складами)'); ax.set_ylabel('медиана индекса модели (внутри склада)'); plt.show()

Значения воспроизводятся из выгрузки GPKG и могут отличаться от сохранённых в БД из-за случайного зерна и версий библиотек.